In [1]:
import time
from pathlib import Path
from github_rag.ingestion.repository import GitHubRepositoryLoader
from github_rag.processing.processor import Processor
from github_rag.chunking.chunker import Chunker
from github_rag.embedding.embedding_model import create_embedding_model
from github_rag.vectordatabase.chroma_store import ChromaStore
from github_rag.retrieval.retriever import Retriever

c:\Users\david\anaconda3\envs\github-rag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
REPOSITORY_URL = "https://github.com/davidfernxndez/spark-active-learning"

## 1. Ingestion -> processing -> chunking

This section apply the Ingestion, processing and chunking phase to get the prepared *LangChain Documents* for Embedding and VectorDB storing.

In [3]:
# Set the Loader Object with the URL
loader = GitHubRepositoryLoader(REPOSITORY_URL)

# Load the repository files as a list of LangChain Documents
documents = loader.load()

print(f"Number of documents ingested: {len(documents)}")

Number of documents ingested: 12


In [4]:
# Set the Processor Object
processor = Processor()

# Apply processing to the ingested documents
documents_processed = processor.process(documents)

print(f"Number of documents before processing: {len(documents_processed)}")

Number of documents before processing: 45


In [5]:
# Set the Chunker object that managed all the specific chunkers
chunker = Chunker()

# Apply chunking
documents_chunking = chunker.chunk(documents_processed)

print(f"Number of documents before chunking: {len(documents_chunking)}")

Number of documents before chunking: 183


## 2. Load the Embedding model

We first need to load the embedding model that will be passed to the ChromaDB instance. The same model used to generate the document embeddings must be used to embed user queries during the retrieval process.

In [6]:
# Set the HuggingFace embedding model name
embedding_model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

# Load the model
embedding_model = create_embedding_model(embedding_model_name)

## 3. Create Chroma Database

The `ChromaStore` class supports two different storage modes depending on the execution environment. For local development, it can create a **persistent Chroma database** stored on disk. The database is stored in the `/data` directory at the root of the repository, allowing it to persist between executions and making it convenient for development and experimentation.

It also supports creating an **in-memory Chroma database**, which is intended for the Streamlit deployment. In this mode, the vector database exists only during the current application session and does not require persistent storage on disk.

In [7]:
start_time = time.time()
# Create ChromaDB instance for local disk storing
vectorDB_local = ChromaStore(
    mode="local",
    collection_name="github_repo"
)

# Create the database with documents and the embedding model
vectorDB_local.create(
    documents=documents_chunking,
    embedding_model=embedding_model
)
print(f"Database generation time: {time.time() - start_time:.2f} seconds")

# Check the number of stored documents
stored_documents = vectorDB_local.count()

print(f"Documents provided: {len(documents_chunking)}")
print(f"Documents stored:   {stored_documents}")

Database generation time: 8.45 seconds
Documents provided: 183
Documents stored:   183


In [8]:
file_paths = vectorDB_local.get_unique_file_paths()

print(file_paths)

['README.md', 'data/data_info.md', 'enviroment.yml', 'main.ipynb', 'performance_experiment.ipynb', 'scalability_experiments.ipynb', 'scripts/AL_methods.py', 'scripts/AL_performance.py', 'scripts/AL_scalability.py', 'scripts/config.py', 'scripts/dowload_data.py', 'scripts/plot_results.py']


## 4. Retrieval

In [49]:
# Query
query = "En que parte de scripts/AL_methods.py se ejecuta el filtrado por incertidumbre?"

print("=" * 100)
print("RETRIEVAL COMPARISON")
print("=" * 100)
print(f"\nQuery: {query}\n")


# ============================================================
# 1. Similarity Search
# ============================================================

results_similarity = vectorDB_local.similarity_search_with_score(
    query=query,
    k=5
)


print("=" * 100)
print("1. SIMILARITY SEARCH")
print("=" * 100)

for i, (document, distance) in enumerate(results_similarity, start=1):

    print(f"\n--- Result {i} ---")
    print(f"Distance : {distance:.4f}")
    print(f"Source   : {document.metadata.get('source')}")
    print(f"File     : {document.metadata.get('file_path')}")
    print(f"\n{document.page_content}")


# ============================================================
# 2. Maximal Marginal Relevance (MMR)
# ============================================================

results_mmr = vectorDB_local.max_marginal_relevance_search(
    query=query,
    k=5,
    fetch_k=20,
    lambda_mult=0.5
)


print("\n\n")
print("=" * 100)
print("2. MAXIMAL MARGINAL RELEVANCE (MMR)")
print("=" * 100)

print("\nParameters:")
print("  k           = 5")
print("  fetch_k     = 20")
print("  lambda_mult = 0.5")

for i, document in enumerate(results_mmr, start=1):

    print(f"\n--- Result {i} ---")
    print(f"Source : {document.metadata.get('source')}")
    print(f"File   : {document.metadata.get('file_path')}")
    print(f"\n{document.page_content}")


# ============================================================
# 3. Summary
# ============================================================

print("\n\n")
print("=" * 100)
print("SUMMARY")
print("=" * 100)

print("\nSimilarity Search:")
for i, (document, distance) in enumerate(results_similarity, start=1):
    print(
        f"{i}. "
        f"{document.metadata.get('file_path')} "
        f"(distance={distance:.4f})"
    )

print("\nMMR:")
for i, document in enumerate(results_mmr, start=1):
    print(
        f"{i}. "
        f"{document.metadata.get('file_path')}"
    )

RETRIEVAL COMPARISON

Query: En que parte de scripts/AL_methods.py se ejecuta el filtrado por incertidumbre?

1. SIMILARITY SEARCH

--- Result 1 ---
Distance : 12.0055
Source   : https://github.com/davidfernxndez/spark-active-learning/blob/main/main.ipynb
File     : main.ipynb

## 3.3 *get_uncertainty_candidates()*  
This method implements uncertainty-based filtering to obtain a subset of candidates from the unlabeled set $U_t$ at iteration $t$. The goal is to focus the subsequent diversity-based selection phase on the potentially most informative instances while reducing the amount of data that needs to be processed.  
To do this, the training DataFrame is filtered to obtain the instances of $U_t$, and the model trained on the labeled instances—obtained via *train_and_evaluate()*—is used to calculate predictions and probabilities for $U_t$. Since this is a binary classification problem, the following uncertainty measure is employed:
$$U(x) = 1 - 2 \left| P(y=1 \mid x) - 0.5 \right|$$


In [9]:
retriever = Retriever(vectorDB_local)



In [38]:
query = "Which are the configuration global variables? @config.py"

results = retriever.retrieve(query=query, available_files=file_paths)

In [39]:
results

[Document(id='8b2acac7-5cfa-48ed-b950-0e60cd00d43a', metadata={'repository': 'davidfernxndez/spark-active-learning', 'file_type': '.py', 'file_path': 'scripts/config.py', 'branch': 'main', 'source': 'https://github.com/davidfernxndez/spark-active-learning/blob/main/scripts/config.py'}, page_content='from pathlib import Path\n\n# ============================================================\n# Project paths\n# ============================================================\n\nPROJECT_ROOT = Path(__file__).resolve().parent.parent\n\nDATA_DIR = PROJECT_ROOT / "data"\nCOMPRESSED_FILE_PATH = DATA_DIR / "HIGGS.csv.gz"\nSCRIPTS_DIR = PROJECT_ROOT / "scripts"\nRESULTS_DIR = PROJECT_ROOT / "results"\n\n# ============================================================\n# Dataset\n# ============================================================\nDATASET_URL = (\n    "https://archive.ics.uci.edu/ml/"\n    "machine-learning-databases/00280/HIGGS.csv.gz"\n)\nTARGET_SAMPLES = 1_000_000\nDATA_PERCENTAGES = [10

In [40]:
context = "\n\n".join(
    f"""File_name: {doc.metadata.get("file_path")}
Source: {doc.metadata.get("source")}

Content:
{doc.page_content}"""
    for doc in results
)

prompt = f"""
Answer the question about the GitHub repository using only the information
provided in the context.

The context contains:
- File_name: the path of the file in the repository.
- Source: the URL of the source file in the repository.
- Content: the relevant content extracted from the file.

Use File_name when referring to a specific file.
Use Source when providing a source reference or citation.

Context:
{context}

Question:
{query}
"""

print(prompt)


Answer the question about the GitHub repository using only the information
provided in the context.

The context contains:
- File_name: the path of the file in the repository.
- Source: the URL of the source file in the repository.
- Content: the relevant content extracted from the file.

Use File_name when referring to a specific file.
Use Source when providing a source reference or citation.

Context:
File_name: scripts/config.py
Source: https://github.com/davidfernxndez/spark-active-learning/blob/main/scripts/config.py

Content:
from pathlib import Path

# ============================================================
# Project paths
# ============================================================

PROJECT_ROOT = Path(__file__).resolve().parent.parent

DATA_DIR = PROJECT_ROOT / "data"
COMPRESSED_FILE_PATH = DATA_DIR / "HIGGS.csv.gz"
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
RESULTS_DIR = PROJECT_ROOT / "results"

# ============================================================
# Dataset
# =

In [14]:
from langchain_ollama import ChatOllama

print("langchain-ollama importado correctamente")

langchain-ollama importado correctamente


In [15]:
llm = ChatOllama(
    model="qwen3:1.7b",
    temperature=0,
)

In [41]:
import time

start = time.time()

response = llm.invoke(prompt)

print(f"Time: {time.time() - start:.2f} seconds")
print(response.content)

Time: 19.26 seconds
The configuration global variables in the `scripts/config.py` file are:

1. **PROJECT_ROOT**  
   - Defined as: `PROJECT_ROOT = Path(__file__).resolve().parent.parent`  
   - Represents the project root directory.

2. **DATA_DIR**  
   - Defined as: `DATA_DIR = PROJECT_ROOT / "data"`  
   - Path to the dataset directory.

3. **COMPRESSED_FILE_PATH**  
   - Defined as: `COMPRESSED_FILE_PATH = DATA_DIR / "HIGGS.csv.gz"`  
   - Path to the compressed dataset file.

4. **SCRIPTS_DIR**  
   - Defined as: `SCRIPTS_DIR = PROJECT_ROOT / "scripts"`  
   - Path to the scripts directory.

5. **RESULTS_DIR**  
   - Defined as: `RESULTS_DIR = PROJECT_ROOT / "results"`  
   - Path to the results directory.

6. **DATASET_URL**  
   - Defined as: `DATASET_URL = ("https://archive.ics.uci.edu/ml/"`  
   - URL to the HIGGS dataset.

7. **TARGET_SAMPLES**  
   - Defined as: `TARGET_SAMPLES = 1_000_000`  
   - Target number of samples for the dataset.

8. **DATA_PERCENTAGES**  
   - Def

In [ ]:
import time

# Insert API KEY from .env

from langchain_groq import ChatGroq
"""
llm_groq = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=GROQ_API_KEY,
    temperature=0,
)
"""

In [52]:
response = llm_groq.invoke(prompt)

print(response.content)

El filtrado por incertidumbre se realiza dentro de la función **`get_uncertainty_candidates()`** en el archivo `scripts/AL_methods.py`.  
En concreto, después de calcular la columna `uncertainty`, se ejecuta la siguiente operación:

```python
candidates_df = (
    unlabeled_pred_df
    .select("id_sample", "features", "uncertainty")
    .filter(sql_f.col("uncertainty") >= threshold_val)
)
```

Esta línea filtra el DataFrame de instancias sin etiquetar (`unlabeled_pred_df`) conservando únicamente aquellas cuyo valor de incertidumbre es mayor o igual al umbral `threshold_val`.  

**Referencia**  
- File_name: `scripts/AL_methods.py`  
- Source: https://github.com/davidfernxndez/spark-active-learning/blob/main/scripts/AL_methods.py
